## ML Problem: "Multi-class classification -- predict job role from skills, experience level, and work type. X classes, Y total samples after filtering."

In [122]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sys
import joblib

import sys
sys.path.append('../src')
from analytics import *

In [123]:
skills_df = pd.read_csv("../data/job_skills_clean.csv")

In [124]:
skills_df["title"]=skills_df["title"].str.title()

In [125]:
skills_df.to_csv("../data/job_skills_clean.csv")

In [126]:
skills_df = pd.read_csv("../data/job_skills_clean.csv")

In [127]:
skills_df["title"].value_counts().head(20)

title
Sales Manager                      1350
Assistant Store Manager             632
Project Manager                     630
Customer Service Representative     573
Salesperson                         492
Senior Accountant                   405
Account Executive                   399
Store Manager                       389
Executive Assistant                 371
Software Engineer                   369
Sales Associate                     367
Business Analyst                    344
Account Manager                     344
Staff Accountant                    342
Controller                          332
Sales Executive                     324
Administrative Assistant            316
Retail Sales Associate              302
Maintenance Technician              300
Business Development Manager        295
Name: count, dtype: int64

In [128]:
skills_df.shape

(204974, 23)

In [129]:
skills_df.head(5)

,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,job_id,company_name,title,description,location,views,...,remote_allowed,application_type,expiry,formatted_experience_level,listed_time,work_type,normalized_salary,clean_location,skill_abr,skill_name
0,0,0,0,0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,"Princeton, NJ",20.0,...,Not specified,ComplexOnsiteApply,2024-05-17,Not specified,2024-04-17,FULL_TIME,38480.0,Princeton,MRKT,Marketing
1,1,1,1,1,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,"Princeton, NJ",20.0,...,Not specified,ComplexOnsiteApply,2024-05-17,Not specified,2024-04-17,FULL_TIME,38480.0,Princeton,SALE,Sales
2,2,2,2,2,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...","Fort Collins, CO",1.0,...,Not specified,ComplexOnsiteApply,2024-05-11,Not specified,2024-04-11,FULL_TIME,83200.0,Fort Collins,HCPR,Health Care Provider
3,3,3,3,3,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,"Cincinnati, OH",8.0,...,Not specified,ComplexOnsiteApply,2024-05-16,Not specified,2024-04-16,FULL_TIME,55000.0,Cincinnati,MGMT,Management
4,4,4,4,4,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,"Cincinnati, OH",8.0,...,Not specified,ComplexOnsiteApply,2024-05-16,Not specified,2024-04-16,FULL_TIME,55000.0,Cincinnati,MNFC,Manufacturing


In [173]:
skills_df["formatted_experience_level"].unique()

array(['Not specified', 'Entry level', 'Mid-Senior level', 'Associate',
       'Director', 'Internship', 'Executive'], dtype=object)

### Defining target column:

In [130]:
skills_df["title"]=skills_df["title"].str.title()

In [131]:
top_roles=get_top_roles(skills_df,30)
top_roles

title
Sales Manager                      1350
Assistant Store Manager             632
Project Manager                     630
Customer Service Representative     573
Salesperson                         492
Senior Accountant                   405
Account Executive                   399
Store Manager                       389
Executive Assistant                 371
Software Engineer                   369
Sales Associate                     367
Business Analyst                    344
Account Manager                     344
Staff Accountant                    342
Controller                          332
Sales Executive                     324
Administrative Assistant            316
Retail Sales Associate              302
Maintenance Technician              300
Business Development Manager        295
Senior Software Engineer            276
Outside Sales Representative        272
General Manager                     266
Assistant Manager                   250
Data Analyst                      

In [132]:
ml_df=skills_df[skills_df["title"].isin(top_roles.index)]

In [133]:
ml_df.head()

,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,job_id,company_name,title,description,location,views,...,remote_allowed,application_type,expiry,formatted_experience_level,listed_time,work_type,normalized_salary,clean_location,skill_abr,skill_name
25,25,25,25,25,115639136,Shannon Waltchack,Controller,WORK @ SWShannon Waltchack (SW) is seeking a C...,"Birmingham, AL",61.0,...,Not specified,ComplexOnsiteApply,2024-10-12,Not specified,2024-04-15,FULL_TIME,NaN,Birmingham,ACCT,Accounting/Auditing
26,26,26,26,26,115639136,Shannon Waltchack,Controller,WORK @ SWShannon Waltchack (SW) is seeking a C...,"Birmingham, AL",61.0,...,Not specified,ComplexOnsiteApply,2024-10-12,Not specified,2024-04-15,FULL_TIME,NaN,Birmingham,MGMT,Management
27,27,27,27,27,115639136,Shannon Waltchack,Controller,WORK @ SWShannon Waltchack (SW) is seeking a C...,"Birmingham, AL",61.0,...,Not specified,ComplexOnsiteApply,2024-10-12,Not specified,2024-04-15,FULL_TIME,NaN,Birmingham,HR,Human Resources
30,30,30,30,30,133130219,NaN,Software Engineer,"Education Bachelor's degree in software, math,...",Los Angeles Metropolitan Area,1.0,...,Not specified,ComplexOnsiteApply,2024-05-19,Not specified,2024-04-19,FULL_TIME,NaN,Los Angeles Metropolitan Area,ENG,Engineering
31,31,31,31,31,133130219,NaN,Software Engineer,"Education Bachelor's degree in software, math,...",Los Angeles Metropolitan Area,1.0,...,Not specified,ComplexOnsiteApply,2024-05-19,Not specified,2024-04-19,FULL_TIME,NaN,Los Angeles Metropolitan Area,IT,Information Technology


In [134]:
ml_df.shape

(11360, 23)

In [135]:
ml_df["title"].nunique()

30

### Select your feature columns:

In [136]:
feature_cols=[
    "skill_name",
    "formatted_experience_level",
    "formatted_work_type",
    "remote_allowed"
]

In [137]:
ml_df[feature_cols].isnull().mean()*100   

skill_name                    0.0
formatted_experience_level    0.0
formatted_work_type           0.0
remote_allowed                0.0
dtype: float64

## Label Encoding:

In [138]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()

In [139]:
y=ml_df["title"]

In [140]:
ml_df["title_encoded"]=le.fit_transform(y)

C:\Users\punit\AppData\Local\Temp\ipykernel_25780\1094493885.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ml_df["title_encoded"]=le.fit_transform(y)


In [141]:
ml_df.head()

,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,job_id,company_name,title,description,location,views,...,application_type,expiry,formatted_experience_level,listed_time,work_type,normalized_salary,clean_location,skill_abr,skill_name,title_encoded
25,25,25,25,25,115639136,Shannon Waltchack,Controller,WORK @ SWShannon Waltchack (SW) is seeking a C...,"Birmingham, AL",61.0,...,ComplexOnsiteApply,2024-10-12,Not specified,2024-04-15,FULL_TIME,NaN,Birmingham,ACCT,Accounting/Auditing,8
26,26,26,26,26,115639136,Shannon Waltchack,Controller,WORK @ SWShannon Waltchack (SW) is seeking a C...,"Birmingham, AL",61.0,...,ComplexOnsiteApply,2024-10-12,Not specified,2024-04-15,FULL_TIME,NaN,Birmingham,MGMT,Management,8
27,27,27,27,27,115639136,Shannon Waltchack,Controller,WORK @ SWShannon Waltchack (SW) is seeking a C...,"Birmingham, AL",61.0,...,ComplexOnsiteApply,2024-10-12,Not specified,2024-04-15,FULL_TIME,NaN,Birmingham,HR,Human Resources,8
30,30,30,30,30,133130219,NaN,Software Engineer,"Education Bachelor's degree in software, math,...",Los Angeles Metropolitan Area,1.0,...,ComplexOnsiteApply,2024-05-19,Not specified,2024-04-19,FULL_TIME,NaN,Los Angeles Metropolitan Area,ENG,Engineering,27
31,31,31,31,31,133130219,NaN,Software Engineer,"Education Bachelor's degree in software, math,...",Los Angeles Metropolitan Area,1.0,...,ComplexOnsiteApply,2024-05-19,Not specified,2024-04-19,FULL_TIME,NaN,Los Angeles Metropolitan Area,IT,Information Technology,27


In [142]:
le.classes_

array(['Account Executive', 'Account Manager', 'Accountant',
       'Administrative Assistant', 'Assistant Manager',
       'Assistant Store Manager', 'Business Analyst',
       'Business Development Manager', 'Controller',
       'Customer Service Representative', 'Data Analyst',
       'Executive Assistant', 'General Manager', 'Maintenance Technician',
       'Mortgage Loan Officer', 'Operations Assistant Manager',
       'Outside Sales Representative', 'Project Manager',
       'Retail Sales Associate', 'Sales Associate', 'Sales Executive',
       'Sales Manager', 'Sales Representative', 'Sales Specialist',
       'Salesperson', 'Senior Accountant', 'Senior Software Engineer',
       'Software Engineer', 'Staff Accountant', 'Store Manager'],
      dtype=object)

In [143]:
le.transform(le.classes_)

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29])

In [144]:
mapping = dict(
    zip(
        le.classes_,
        le.transform(le.classes_).tolist()  ## convert into simple list from numpy list to save in json format
    )
)
mapping

{'Account Executive': 0,
 'Account Manager': 1,
 'Accountant': 2,
 'Administrative Assistant': 3,
 'Assistant Manager': 4,
 'Assistant Store Manager': 5,
 'Business Analyst': 6,
 'Business Development Manager': 7,
 'Controller': 8,
 'Customer Service Representative': 9,
 'Data Analyst': 10,
 'Executive Assistant': 11,
 'General Manager': 12,
 'Maintenance Technician': 13,
 'Mortgage Loan Officer': 14,
 'Operations Assistant Manager': 15,
 'Outside Sales Representative': 16,
 'Project Manager': 17,
 'Retail Sales Associate': 18,
 'Sales Associate': 19,
 'Sales Executive': 20,
 'Sales Manager': 21,
 'Sales Representative': 22,
 'Sales Specialist': 23,
 'Salesperson': 24,
 'Senior Accountant': 25,
 'Senior Software Engineer': 26,
 'Software Engineer': 27,
 'Staff Accountant': 28,
 'Store Manager': 29}

In [145]:
import json

with open("../data/title_mapping.json", "w") as f:
    json.dump(mapping, f, indent=4)

In [146]:
import pickle

with open("../models/label_encoder.pkl","wb") as f:
    pickle.dump(le,f)
    

## Class Distribution:

In [147]:
class_counts = ml_df["title"].value_counts()
class_counts

title
Sales Manager                      1350
Assistant Store Manager             632
Project Manager                     630
Customer Service Representative     573
Salesperson                         492
Senior Accountant                   405
Account Executive                   399
Store Manager                       389
Executive Assistant                 371
Software Engineer                   369
Sales Associate                     367
Account Manager                     344
Business Analyst                    344
Staff Accountant                    342
Controller                          332
Sales Executive                     324
Administrative Assistant            316
Retail Sales Associate              302
Maintenance Technician              300
Business Development Manager        295
Senior Software Engineer            276
Outside Sales Representative        272
General Manager                     266
Assistant Manager                   250
Data Analyst                      

In [148]:
print("Number of classes:", ml_df["title"].nunique())
print("Total samples:", len(ml_df))
print("Largest class:", class_counts.max())
print("Smallest class:", class_counts.min())

Number of classes: 30
Total samples: 11360
Largest class: 1350
Smallest class: 228


## Feature engineering:

#### One hot encoding on skill column:

In [149]:
ml_df.shape

(11360, 24)

In [150]:
ml_df["skill_name"].value_counts()

skill_name
Sales                     2747
Business Development      2392
Accounting/Auditing        733
Information Technology     703
Finance                    586
Management                 573
Other                      568
Administrative             537
Customer Service           494
Manufacturing              375
Project Management         339
Engineering                338
Analyst                    234
Research                   157
General Business           115
Advertising                 92
Marketing                   88
Training                    73
Consulting                  57
Strategy/Planning           22
Human Resources             18
Production                  15
Supply Chain                15
Product Management          15
Health Care Provider        15
Distribution                11
Design                      10
Science                     10
Quality Assurance            6
Public Relations             6
Art/Creative                 5
Legal                       

In [151]:
skill_matrix=pd.crosstab(ml_df["job_id"], ml_df["skill_name"])
skill_matrix = (skill_matrix > 0).astype(int)
skill_matrix = skill_matrix.reset_index()   ## to get job_id as col instead of index
skill_matrix

skill_name,job_id,Accounting/Auditing,Administrative,Advertising,Analyst,Art/Creative,Business Development,Consulting,Customer Service,Design,...,Public Relations,Purchasing,Quality Assurance,Research,Sales,Science,Strategy/Planning,Supply Chain,Training,Writing/Editing
0,115639136,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,133130219,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,175485704,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,229924287,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,258868844,0,0,0,0,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6087,3906262087,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6088,3906262103,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6089,3906263103,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6090,3906265319,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


In [152]:
job_ml_df=ml_df.drop_duplicates(subset=["job_id"])   ## removing duplicates job_id for job-skill pair (onl one job_id for all skill)
job_ml_df

,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,job_id,company_name,title,description,location,views,...,application_type,expiry,formatted_experience_level,listed_time,work_type,normalized_salary,clean_location,skill_abr,skill_name,title_encoded
25,25,25,25,25,115639136,Shannon Waltchack,Controller,WORK @ SWShannon Waltchack (SW) is seeking a C...,"Birmingham, AL",61.0,...,ComplexOnsiteApply,2024-10-12,Not specified,2024-04-15,FULL_TIME,NaN,Birmingham,ACCT,Accounting/Auditing,8
30,30,30,30,30,133130219,NaN,Software Engineer,"Education Bachelor's degree in software, math,...",Los Angeles Metropolitan Area,1.0,...,ComplexOnsiteApply,2024-05-19,Not specified,2024-04-19,FULL_TIME,NaN,Los Angeles Metropolitan Area,ENG,Engineering,27
35,35,35,35,35,175485704,GOYT,Software Engineer,Job Description:GOYT is seeking a skilled and ...,"Denver, CO",273.0,...,ComplexOnsiteApply,2024-05-16,Not specified,2024-04-16,PART_TIME,NaN,Denver,ENG,Engineering,27
40,40,40,40,40,229924287,REquipment Durable Medical Equipment and Assis...,Administrative Assistant,The Administrative Assistant will organize and...,"Woburn, MA",3.0,...,ComplexOnsiteApply,2024-05-19,Not specified,2024-04-19,PART_TIME,47840.0,Woburn,ADM,Administrative,3
44,44,44,44,44,258868844,ABME,Salesperson,"ABME, a leader in corporate event planning, is...",United States,4.0,...,ComplexOnsiteApply,2024-05-18,Not specified,2024-04-18,FULL_TIME,NaN,United States,SALE,Sales,24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204856,204856,204856,204856,204856,3906262087,Mindlance,Project Manager,Title: Project ManagerDuration: 4 Months – Lon...,"Washington, DC",2.0,...,ComplexOnsiteApply,2024-05-19,Mid-Senior level,2024-04-19,CONTRACT,130000.0,Washington,IT,Information Technology,17
204861,204861,204861,204861,204861,3906262103,Beacon Hill Staffing Group,Staff Accountant,Are you looking for the opportunity to jump on...,"Portland, OR",NaN,...,SimpleOnsiteApply,2024-05-19,Entry level,2024-04-19,FULL_TIME,NaN,Portland,ACCT,Accounting/Auditing,28
204890,204890,204890,204890,204890,3906263103,Rose International,Software Engineer,Date Posted: 04/19/2024Hiring Organization: Ro...,"Mossville, IL",2.0,...,ComplexOnsiteApply,2024-05-19,Mid-Senior level,2024-04-19,TEMPORARY,66560.0,Mossville,IT,Information Technology,27
204944,204944,204944,204944,204944,3906265319,pepperl+fuchs,Account Manager,Are you looking for a career that allows you t...,"Milwaukee, WI",1.0,...,ComplexOnsiteApply,2024-05-20,Mid-Senior level,2024-04-20,FULL_TIME,NaN,Milwaukee,MNFC,Manufacturing,1


In [153]:
job_ml_df =job_ml_df[["job_id","title","title_encoded","formatted_experience_level","formatted_work_type","remote_allowed"]]
final_df=job_ml_df.merge(skill_matrix,on="job_id")
final_df

,job_id,title,title_encoded,formatted_experience_level,formatted_work_type,remote_allowed,Accounting/Auditing,Administrative,Advertising,Analyst,...,Public Relations,Purchasing,Quality Assurance,Research,Sales,Science,Strategy/Planning,Supply Chain,Training,Writing/Editing
0,115639136,Controller,8,Not specified,Full-time,Not specified,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,133130219,Software Engineer,27,Not specified,Full-time,Not specified,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,175485704,Software Engineer,27,Not specified,Part-time,Not specified,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,229924287,Administrative Assistant,3,Not specified,Part-time,Not specified,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,258868844,Salesperson,24,Not specified,Full-time,Yes,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6087,3906262087,Project Manager,17,Mid-Senior level,Contract,Not specified,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6088,3906262103,Staff Accountant,28,Entry level,Full-time,Not specified,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6089,3906263103,Software Engineer,27,Mid-Senior level,Temporary,Not specified,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6090,3906265319,Account Manager,1,Mid-Senior level,Full-time,Not specified,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


#### One hot encoding on formatted_work_type:

In [174]:
final_df["formatted_experience_level"].value_counts()

formatted_experience_level
3.0    2477
1.0    1673
0.0    1111
2.0     593
4.0     163
5.0      75
Name: count, dtype: int64

In [154]:
final_df["formatted_work_type"].value_counts()

formatted_work_type
Full-time    5009
Part-time     560
Contract      487
Temporary      19
Other          15
Volunteer       2
Name: count, dtype: int64

In [155]:
final_df["formatted_work_type"]=final_df["formatted_work_type"].replace(["Temporary","Other","Volunteer"],"Other")
final_df["formatted_work_type"].value_counts()

formatted_work_type
Full-time    5009
Part-time     560
Contract      487
Other          36
Name: count, dtype: int64

In [156]:
final_df["remote_allowed"].value_counts()

remote_allowed
Not specified    4809
Yes              1283
Name: count, dtype: int64

In [157]:
final_df=pd.get_dummies(final_df,columns=["formatted_work_type","remote_allowed"],drop_first=True)
final_df

,job_id,title,title_encoded,formatted_experience_level,Accounting/Auditing,Administrative,Advertising,Analyst,Art/Creative,Business Development,...,Sales,Science,Strategy/Planning,Supply Chain,Training,Writing/Editing,formatted_work_type_Full-time,formatted_work_type_Other,formatted_work_type_Part-time,remote_allowed_Yes
0,115639136,Controller,8,Not specified,1,0,0,0,0,0,...,0,0,0,0,0,0,True,False,False,False
1,133130219,Software Engineer,27,Not specified,0,0,0,0,0,0,...,0,0,0,0,0,0,True,False,False,False
2,175485704,Software Engineer,27,Not specified,0,0,0,0,0,0,...,0,0,0,0,0,0,False,False,True,False
3,229924287,Administrative Assistant,3,Not specified,0,1,0,0,0,0,...,0,0,0,0,0,0,False,False,True,False
4,258868844,Salesperson,24,Not specified,0,0,0,0,0,1,...,1,0,0,0,0,0,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6087,3906262087,Project Manager,17,Mid-Senior level,0,0,0,0,0,0,...,0,0,0,0,0,0,False,False,False,False
6088,3906262103,Staff Accountant,28,Entry level,1,0,0,0,0,0,...,0,0,0,0,0,0,True,False,False,False
6089,3906263103,Software Engineer,27,Mid-Senior level,0,0,0,0,0,0,...,0,0,0,0,0,0,False,True,False,False
6090,3906265319,Account Manager,1,Mid-Senior level,0,0,0,0,0,0,...,1,0,0,0,0,0,True,False,False,False


#### ordinal encoding on formatted_work_experience:

In [158]:
final_df["formatted_experience_level"].value_counts()

formatted_experience_level
Mid-Senior level    2477
Entry level         1673
Not specified       1111
Associate            593
Director             163
Executive             75
Name: count, dtype: int64

In [159]:
from sklearn.preprocessing import OrdinalEncoder

In [160]:
final_df.shape

(6092, 43)

In [161]:
oe=OrdinalEncoder(categories=[["Not specified","Entry level","Associate","Mid-Senior level","Director","Executive"]])

In [163]:
oe.categories

[['Not specified',
  'Entry level',
  'Associate',
  'Mid-Senior level',
  'Director',
  'Executive']]

In [171]:
joblib.dump(oe, "C:/Users/punit/OneDrive/Documents/Job_market_intelligence_project/models/experience_encoder.pkl")

['C:/Users/punit/OneDrive/Documents/Job_market_intelligence_project/models/experience_encoder.pkl']

In [168]:
final_df["formatted_experience_level"]=oe.fit_transform(final_df[["formatted_experience_level"]])

ValueError: could not convert string to float: 'Not specified'

In [ ]:
final_df

In [169]:
final_df.to_csv("../data/final_df.csv",index=False)

## X and y split:

In [170]:
X = final_df.drop(columns=["job_id","title","title_encoded"])
y = final_df["title_encoded"]

In [ ]:
X.head()

In [ ]:
feature_columns = X.columns.tolist()

import json

with open("C:/Users/punit/OneDrive/Documents/Job_market_intelligence_project/models/feature_columns.json", "w") as f:
    json.dump(feature_columns, f)

In [ ]:
y.head()

In [ ]:
print(X.shape)
print(y.shape)

## Train-Test split:

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

In [ ]:
## checking distribution of classes after split

In [ ]:
(y_train.value_counts(normalize=True)*100).head()

In [ ]:
(y_test.value_counts(normalize=True)*100).head()

# Training the model

## (1) Logistic Regression:

In [ ]:
from sklearn.linear_model import LogisticRegression

model=LogisticRegression(max_iter=1000,random_state=42)
model.fit(X_train, y_train)


In [ ]:
print("Model trained successfully")

In [ ]:
y_pred=model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score
accuracy=accuracy_score(y_test,y_pred)
print("Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
le.classes_

In [ ]:
for i, role in enumerate(le.classes_):
    print(i, role)

## Logistic Regression Results

The Logistic Regression model achieved an accuracy of 60.13%, substantially outperforming random guessing (~3.3% for 30 classes).

The model performed best on specialized roles such as Mortgage Loan Officer, Project Manager, and Operations Assistant Manager. These roles have distinctive skill requirements that make them easier to identify.

The model struggled with several sales-related roles including Sales Representative, Sales Specialist, and Outside Sales Representative. These job titles share very similar skill profiles, leading to frequent misclassification.

Overall, skills, experience level, work type, and remote status provide strong predictive power for job role classification.

In [ ]:
print("Top skills for Sales Representive:" ,get_role_skills(skills_df,"Sales Representative"),"\n")
print("Top skills for Sales Specialist:" ,get_role_skills(skills_df,"Sales Specialist"),"\n")
print("Top skills for Outside Sales Representive:" ,get_role_skills(skills_df,"Outside Sales Representative"))

All three of them have sales and business devlopment as top skills which make it difficult to distinguish between them.

## (2) Random Forest:

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier(n_estimators=100,random_state=42)

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
rf.fit(X_train,y_train)

In [ ]:
y_pred_rf=rf.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_rf=accuracy_score(y_test,y_pred_rf)
print("Random Forest Accuracy:", accuracy_rf)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_rf))

#### Feature importance:

In [ ]:
rf.feature_importances_

In [ ]:
importances = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
)
importances.sort_values(ascending=False).head(15)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
importances.sort_values(ascending=False).head(15).plot(kind="barh")
plt.xlabel("Feature Importance")
plt.title("Top 15 Important Features")
plt.gca().invert_yaxis()
plt.show()

## confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm=confusion_matrix(y_test,y_pred_rf)
plt.figure(figsize=(14,12))
sns.heatmap(cm,cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Random Forest")
plt.show()


## Compariosn Table:

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
lr_report=classification_report(y_test,y_pred,output_dict=True)
rf_report=classification_report(y_test,y_pred_rf,output_dict=True)

In [ ]:
lr_report["macro avg"]

In [ ]:
rf_report["macro avg"]

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1"
    ],
    "Logistic Regression": [
        accuracy,
        lr_report["macro avg"]["precision"],
        lr_report["macro avg"]["recall"],
        lr_report["macro avg"]["f1-score"]
    ],
    "Random Forest": [
        accuracy_rf,
        rf_report["macro avg"]["precision"],
        rf_report["macro avg"]["recall"],
        rf_report["macro avg"]["f1-score"]
    ]
})

comparison

## Model Comparison & Final Decision

Random Forest outperforms Logistic Regression across all metrics:
- Accuracy: 66.9% vs 60.1%
- Macro F1: 57.3% vs 50.8%

Decision: Random Forest selected as the final model for this project.
Reason: Better handles non-linear interactions between skill, experience, 
and work type features. Consistently stronger across precision, recall, 
and F1 — not just raw accuracy.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sns.heatmap(confusion_matrix(y_test,y_pred),cmap="Blues",ax=axes[0])
axes[0].set_title("Logistic Regression")

sns.heatmap(confusion_matrix(y_test,y_pred_rf),cmap="Blues",ax=axes[1])
axes[1].set_title("Random Forest")

plt.show()

#### Identifying the hardest-to-predict roles:

In [ ]:
print(classification_report(y_test,y_pred_rf,target_names=le.classes_))

In [ ]:
### sales specialist andsales executice were hardest to predict

### Hyperparameter Tuning:

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

#### Manual grid search:

In [ ]:
results=[]

for n in [50,100,200]:
    for depth in [None,10,20]:

        rf=RandomForestClassifier(n_estimators=n,max_depth=depth,random_state=42)
        rf.fit(X_train,y_train)
        y_pred=rf.predict(X_test)

        acc=accuracy_score(y_test,y_pred)

        results.append([n,depth,acc])

print(results)

In [ ]:
results_df=pd.DataFrame(results,columns=["n_estimators","max_depth","accuracy"])

In [ ]:
results_df.sort_values(
    by="accuracy",
    ascending=False
)

## Manual Grid Search Report
To improve the Random Forest model, I manually experimented with different combinations of the hyperparameters `n_estimators` (number of trees) and `max_depth` (maximum depth of each tree). Each model was trained on the same training dataset and evaluated on the test dataset using accuracy.

##### Hyperparameter Combinations Tested
- `n_estimators`: 50, 100, 200
- `max_depth`: None, 10, 20

A total of **9 Random Forest models** were trained and compared.

##### Observations
- The highest accuracy (**67.19%**) was achieved with both:
  - `n_estimators = 50`, `max_depth = 20`
  - `n_estimators = 200`, `max_depth = 20`
- Limiting the tree depth to **10** reduced the model's accuracy, indicating underfitting.
- Increasing the number of trees from **50 to 200** did not improve accuracy for this dataset, suggesting that 50 trees were sufficient.

##### Conclusion
Based on the manual grid search, a **maximum depth of 20** produced the best performance. Since both 50 and 200 trees achieved the same accuracy, the model with **50 trees** is a more computationally efficient choice while maintaining identical predictive performance.

# cross validation:

In [ ]:
best_rf=RandomForestClassifier(n_estimators=50,max_depth=20,random_state=42)

scores=cross_val_score(best_rf,X,y,cv=5,scoring="accuracy")

print("Cross Validation Scores:", scores)
print("Mean Accuracy:", scores.mean())
print("Standard Deviation:", scores.std())

On average, the Random Forest model achieves approximately 63.8% accuracy on unseen data.

## Final model:

In [ ]:
final_rf = RandomForestClassifier(n_estimators=50,max_depth=20,random_state=42)

In [ ]:
final_rf.fit(X_train, y_train)

In [ ]:
y_pred_final = final_rf.predict(X_test)

In [ ]:
final_accuracy=accuracy_score(y_test,y_pred_final)
print("Final Accuracy:", final_accuracy)

#### classifiaction report

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred_final,
        target_names=le.classes_
    )
)

#### confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))

sns.heatmap(
    confusion_matrix(y_test, y_pred_final),
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Final Random Forest Confusion Matrix")

plt.show()

#### saving the model:

In [ ]:
import pickle

with open("../models/job_predictor.pkl","wb") as f:
    pickle.dump(final_rf,f)

print("model saved successfully")

###### Final Model
After selecting the best hyperparameters through manual grid search and validating the model using 5-fold cross-validation, the final Random Forest model was trained using the complete training dataset.

###### Final Hyperparameters
- n_estimators = 50
- max_depth = 20
- random_state = 42

The trained model was evaluated on the test set using accuracy, classification report, and confusion matrix. Finally, the model was saved as `job_predictor.pkl` for deployment in the Streamlit application.

### Analyze Misclassified Samples:

In [ ]:
misclassified = X_test[y_pred_final != y_test]
misclassified

### Per Role F1 Analysis:

In [ ]:
report=classification_report(y_test,y_pred_final,target_names=le.classes_,output_dict=True)
report_df=pd.DataFrame(report).transpose()
report_df

In [ ]:
report_df=report_df[:-3]
report_df ## keeping only classes

In [ ]:
bottom5=report_df.sort_values('f1-score').head(5)
bottom5

In [ ]:
top5=report_df.sort_values("f1-score").tail(5)
top5

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1,2,figsize=(14,5))

axes[0].barh(
    top5.index,
    top5["f1-score"]
)

axes[0].set_title("Top 5 Roles")

axes[1].barh(
    bottom5.index,
    bottom5["f1-score"]
)
axes[0].set_xlabel("F1 Score")
axes[1].set_xlabel("F1 Score")

axes[0].set_ylabel("Job Role")
axes[1].set_ylabel("Job Role")

axes[1].set_title("Bottom 5 Roles")

plt.tight_layout()

plt.show()

### Per-Role F1 Score Analysis

The F1-score analysis highlights the model's strengths and weaknesses across different job roles.

- Roles such as **Mortgage Loan Officer**, **Sales Manager**, and **Project Manager** achieved the highest F1-scores, indicating that the model can classify these roles accurately and consistently.
- In contrast, **Sales Specialist**, **Sales Representative**, and **Sales Associate** obtained the lowest F1-scores. These roles share overlapping skills and responsibilities with other sales-related positions, making them more difficult for the model to distinguish.
- Overall, the model performs well on roles with distinct feature patterns but struggles with classes that have similar skill requirements.

## Test on Manual Profiles:

In [ ]:
X_train.columns

In [ ]:
sample = X_test.iloc[[0]].copy()
sample

In [ ]:
prediction = final_rf.predict(sample)
print(le.inverse_transform(prediction))

In [ ]:
print(y_test.index)

In [ ]:
actual=y_test.loc[sample.index]
print("Actual:", le.inverse_transform(actual)[0])

In [ ]:
sample = X_test.iloc[[5]].copy()
sample

In [ ]:
prediction = final_rf.predict(sample)
print(le.inverse_transform(prediction))

In [ ]:
actual=y_test.loc[sample.index]
print("Actual:", le.inverse_transform(actual)[0])

# Model Card:

## 1. Model Overview

**Model Name:** Random Forest Classifier

**Objective:**  
Predict the job role based on features extracted from LinkedIn job postings.

---

## 2. Dataset

- Source: LinkedIn Job Postings Dataset
- Target Variable: Job Title
- Features Used:
  - Experience Level
  - Job Functions (One-Hot Encoded)
  - Other engineered categorical features
- Train-Test Split: 80% Training / 20% Testing

---

## 3. Model Configuration

| Hyperparameter | Value |
|----------------|-------|
| n_estimators | 50 |
| max_depth | 20 |
| random_state | 42 |

---

## 4. Performance

### Test Accuracy
**67.19%**

### Cross Validation

- Mean Accuracy: **63.77%**
- Standard Deviation: **1.52%**

### Evaluation Metrics

- Accuracy
- Precision
- Recall
- F1-Score
- Confusion Matrix

---

## 5. Strengths

- Performs well on common job roles.
- Captures non-linear relationships between features.
- Stable performance across cross-validation folds.
- Handles categorical features effectively after preprocessing.

---

## 6. Limitations

- Lower performance on classes with fewer training samples.
- Confuses job roles with highly overlapping skills.
- Performance depends on the quality of feature engineering.
- Not suitable for making real hiring decisions.

---

## 7. Intended Use

This model is designed for educational purposes and demonstrates machine learning techniques for predicting job roles from job posting features.

---

## 8. Future Improvements

- Collect more balanced training data.
- Tune additional hyperparameters.
- Try Gradient Boosting (XGBoost, LightGBM).
- Improve feature engineering.
- Deploy using Streamlit for interactive predictions.

## Predict function:

In [1]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

from predict import predict

In [2]:
print(type(model))

NameError: name 'model' is not defined

In [ ]:
sample = {
    "Sales": 1,
    "Management": 1,
    "experience_encoded": 4,
    "formatted_work_type_Full-time": 1
}
print(predict(sample))

## Model Deployment Preparation

To prepare the project for deployment, all required model artifacts were saved in the `models/` directory.

### Saved Files
- `job_predictor.pkl` – Trained Random Forest model
- `label_encoder.pkl` – Label encoder used to convert predictions back to job titles
- `feature_columns.json` – Ordered list of feature names used during training

A reusable prediction module (`predict.py`) was created inside the `src/` directory. The module loads all saved artifacts and exposes a single `predict(user_input)` function, which accepts a dictionary of feature values and returns a human-readable job title.

This modular design separates the machine learning logic from the user interface, making the project easier to maintain and ready for Streamlit integration.

In [ ]:
model = joblib.load("C:/Users/punit/OneDrive/Documents/Job_market_intelligence_project/models/job_predictor.pkl")
label_encoder = joblib.load("C:/Users/punit/OneDrive/Documents/Job_market_intelligence_project/models/label_encoder.pkl")

In [ ]:
print(type(model))

In [ ]:
print(model.feature_names_in_)